# EXPLORA 06 — Rutas, archivos en lote y SQLite

Practica guiada. El profesor va por el `_Caso`, tu por el `_Base`.

Objetivo de la sesion: dejar de abrir archivos a mano.

In [ ]:
from pathlib import Path
import pandas as pd
import sqlite3

DATOS = Path("../../_RECURSOS/datasets/ventas_mensuales")
print(DATOS.resolve())
print("Existe:", DATOS.exists())

## 1 — `Path` en vez de strings

`"carpeta" + "\\" + "archivo.csv"` funciona en Windows y se rompe en Linux, donde el
separador es `/`. En la nube tu codigo corre en Linux. `Path` usa el separador correcto
en cada sistema, solo.

Y ademas trae gratis todo esto:

In [ ]:
archivo = DATOS / "ventas_2025-03.csv"     # el / se usa para unir rutas

print("Ruta completa :", archivo)
print("Nombre        :", archivo.name)
print("Sin extension :", archivo.stem)
print("Extension     :", archivo.suffix)
print("Carpeta padre :", archivo.parent.name)
print("Existe        :", archivo.exists())
print("Tamano (KB)   :", round(archivo.stat().st_size / 1024, 1))

## 2 — Listar la carpeta

`glob("*.csv")` filtra por patron. `sorted()` porque el orden que devuelve el sistema
de archivos no esta garantizado.

`rglob` hace lo mismo pero entrando en subcarpetas.

In [ ]:
archivos = sorted(DATOS.glob("*.csv"))
print(f"{len(archivos)} archivos:")
for a in archivos:
    print(f"  {a.name:24} {a.stat().st_size / 1024:6.1f} KB")

## 3 — Leer uno

In [ ]:
enero = pd.read_csv(archivos[0], sep=";", decimal=",", encoding="utf-8")
print(enero.shape)
enero.head(3)

## 4 — Leer todos

**El patron que hay que memorizar:** lista + un solo `concat`.

`pd.concat` dentro del bucle copia todo lo acumulado en cada vuelta — es O(n²).
Con 12 archivos da igual; con 5.000 no termina nunca.

In [ ]:
partes = []
for archivo in archivos:
    df = pd.read_csv(archivo, sep=";", decimal=",", encoding="utf-8")
    partes.append(df)

todo = pd.concat(partes, ignore_index=True)
print("Consolidado:", todo.shape)

## 5 — Trazabilidad

Cuando un mes venga mal —y va a pasar— esta columna te dice cual reprocesar.
Cuesta una linea. Siempre se pone.

In [ ]:
partes = []
for archivo in archivos:
    df = pd.read_csv(archivo, sep=";", decimal=",", encoding="utf-8")
    df["archivo_origen"] = archivo.name
    partes.append(df)

todo = pd.concat(partes, ignore_index=True)
print(todo["archivo_origen"].value_counts().sort_index().to_string())

## 6 — Escribir texto

Dos formas. `write_text` para algo corto; `with open` cuando escribes por partes.

`with` cierra el archivo aunque el codigo de adentro falle. Sin `with`, un error deja el
archivo abierto y en Windows nadie mas lo puede tocar.

In [ ]:
resumen = Path("resumen_carga.txt")
lineas = [f"Archivos procesados: {len(archivos)}",
          f"Filas totales      : {len(todo)}",
          ""]
for nombre, n in todo["archivo_origen"].value_counts().sort_index().items():
    lineas.append(f"  {nombre:24} {n:4d} filas")

resumen.write_text("\n".join(lineas), encoding="utf-8")
print(resumen.read_text(encoding="utf-8"))

In [ ]:
# La otra forma, equivalente:
with open("resumen_carga.txt", "a", encoding="utf-8") as f:
    f.write("\n\n-- fin del reporte --\n")

print(Path("resumen_carga.txt").read_text(encoding="utf-8")[-40:])

## 7 — SQLite

SQLite es una base de datos entera en **un archivo**. Sin servidor, sin instalar nada:
viene con Python. Perfecta para prototipos y para aprender SQL.

In [ ]:
todo["fecha"] = pd.to_datetime(todo["fecha"], format="mixed", dayfirst=True)

con = sqlite3.connect("demo_tienda.db")
todo.to_sql("ventas", con, if_exists="replace", index=False)

print("Tablas en la base:")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con).to_string(index=False))

`if_exists="replace"` borra la tabla y la vuelve a crear. Con `"append"`, correr la celda
dos veces te deja el doble de filas **sin ningun aviso**. Es el bug clasico de un proceso
mensual.

## 8 — Consultar

El mismo `groupby` de pandas, en el otro idioma. Vale la pena verlos lado a lado.

In [ ]:
q = """
SELECT ciudad,
       COUNT(*)               AS n_ventas,
       ROUND(SUM(importe), 2) AS total
FROM ventas
WHERE ciudad IS NOT NULL AND ciudad != ''
GROUP BY ciudad
ORDER BY total DESC
"""
print(pd.read_sql(q, con).to_string(index=False))

In [ ]:
# Lo mismo en pandas:
print(todo.dropna(subset=["ciudad"])
      .groupby("ciudad")["importe"]
      .agg(n_ventas="count", total="sum").round(2)
      .sort_values("total", ascending=False).to_string())

## 9 — Cerrar

En Windows, una conexion abierta deja el `.db` bloqueado y no lo puedes ni borrar.

In [ ]:
con.close()
print("Conexion cerrada")

Mejor todavia: `with`, que cierra solo.

In [ ]:
with sqlite3.connect("demo_tienda.db") as con:
    n = pd.read_sql("SELECT COUNT(*) AS n FROM ventas", con)["n"][0]
print("Filas en la tabla:", n)

---
### Lo que hay que llevarse

1. **Rutas con `Path`**, nunca concatenando strings.
2. **`sorted(carpeta.glob("*.csv"))`** — nunca nombres a mano.
3. **Acumular en lista, un solo `concat`.**
4. **`archivo_origen` siempre.**
5. **`with`** para archivos y para conexiones.
6. **`if_exists="replace"`** si el proceso se va a repetir.